# Phase 3b — XGBoost Ranking
Input: data/processed/candidates.parquet + {user,item,user_item}_features.parquet (from 03_model.ipynb)

## Cell 1: Load candidates + label the training target
For each (user, candidate) pair from `03_model.ipynb`'s val-scoped candidate pool: positive = the candidate item receives an addtocart or transaction event from that user in val (the future window relative to train); negative = every other retrieved candidate ("simple negative sampling" per the roadmap — keep all positives, negatives are just the rest of the same candidate pool, no additional subsampling).

In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

PROCESSED_DIR = Path("../data/processed")

candidates = pd.read_parquet(PROCESSED_DIR / "candidates.parquet")
val = pd.read_parquet(PROCESSED_DIR / "val.parquet")

print("candidates:", candidates.shape)
print("candidate users:", candidates["visitorid"].nunique())

val_target_events = val[val["event"].isin(["addtocart", "transaction"])]
pos_df = val_target_events[["visitorid", "itemid"]].drop_duplicates()
pos_df["label"] = 1

candidates = candidates.merge(pos_df, on=["visitorid", "itemid"], how="left")
candidates["label"] = candidates["label"].fillna(0).astype(int)

print("\nlabel distribution:\n", candidates["label"].value_counts())
print("positive rate:", candidates["label"].mean())
print("users with >=1 positive candidate:", candidates.loc[candidates["label"] == 1, "visitorid"].nunique(),
      "/", candidates["visitorid"].nunique())

candidates: (762000, 7)
candidate users: 15240

label distribution:
 label
0    761774
1       226
Name: count, dtype: int64
positive rate: 0.00029658792650918633
users with >=1 positive candidate: 156 / 15240


## Cell 2: Join features, leakage check
Join the labeled candidates with the train-only USER/ITEM/USER-ITEM feature tables from `03_model.ipynb`. All of those were already asserted train-only (< T1) in that notebook's Cell 5; the check here re-verifies the join didn't introduce anything from val/test.

In [2]:
user_features = pd.read_parquet(PROCESSED_DIR / "user_features.parquet")
item_features = pd.read_parquet(PROCESSED_DIR / "item_features.parquet")
user_item_features = pd.read_parquet(PROCESSED_DIR / "user_item_features.parquet")

training_set = candidates.merge(user_features, on="visitorid", how="left")
training_set = training_set.merge(item_features, on="itemid", how="left", suffixes=("", "_item"))
training_set = training_set.merge(user_item_features, on=["visitorid", "itemid"], how="left")

# user_item_* counts are genuinely 0 (not unknown) when this candidate was never touched in train
for col in ["user_item_views", "user_item_carts", "user_item_purchases"]:
    training_set[col] = training_set[col].fillna(0)
# user_item_days_since_last_interaction / category_affinity stay NaN (no train interaction to
# measure) -- left for XGBoost's native missing-value handling, same policy as Cell 3/4.

# categoryid columns are stored as strings (raw property values) -> numeric for the model
training_set["user_top_category"] = pd.to_numeric(training_set["user_top_category"], errors="coerce")
training_set["item_category"] = pd.to_numeric(training_set["item_category"], errors="coerce")

# candidate_source is the RANKING-ONLY categorical signal -> ordinal code
source_map = {"als": 0, "itemcf": 1, "both": 2}
training_set["candidate_source_code"] = training_set["candidate_source"].map(source_map)

print("training_set shape:", training_set.shape)
print("\nnull rate per column:\n", training_set.isnull().mean().sort_values(ascending=False))

# Leakage check: every recency feature must be non-negative relative to T1 (train-only by construction)
assert (training_set["user_days_since_last_event"].dropna() >= 0).all(), "negative user recency -> leakage"
assert (training_set["item_days_since_first_seen"].dropna() >= 0).all(), "negative item recency -> leakage"
assert (training_set["user_item_days_since_last_interaction"].dropna() >= 0).all(), "negative user-item recency -> leakage"
print("\nleakage check: all recency features non-negative relative to T1 -> PASSED")

print(f"\npositive: {(training_set['label']==1).sum():,}   negative: {(training_set['label']==0).sum():,}")

training_set shape: (762000, 29)

null rate per column:
 category_affinity                        0.948420
user_item_days_since_last_interaction    0.943848
item_purchase_rate                       0.571089
item_cart_rate                           0.296757
user_top_category                        0.077100
item_category                            0.055728
candidate_score                          0.000000
label                                    0.000000
candidate_source                         0.000000
itemid                                   0.000000
als_score                                0.000000
itemcf_score                             0.000000
visitorid                                0.000000
user_unique_items_viewed                 0.000000
user_total_purchases                     0.000000
user_total_carts                         0.000000
user_total_views                         0.000000
rank                                     0.000000
item_total_views                         0.

## Cell 3: Train XGBoost
`objective="binary:logistic"`, `max_depth=5`, `n_estimators=200`, `learning_rate=0.1` — mid-range of the roadmap's specified bands (shallow trees, small ensemble, matches the ~50-candidate scale). No `scale_pos_weight` or other imbalance handling beyond what's specified — the label set is heavily skewed (226 positive / 761,774 negative) because relevant events are genuinely rare in this data, not a bug; adding untested balancing knobs here would be exactly the "unnecessary hyperparameter search" the roadmap rules out. Ranking quality is judged in Phase 4 by Recall/NDCG@10 on held-out data, not by this training accuracy.

In [3]:
import xgboost as xgb

FEATURE_COLS = [
    # USER
    "user_total_views", "user_total_carts", "user_total_purchases",
    "user_unique_items_viewed", "user_top_category",
    "user_days_since_last_event", "user_interaction_span_days",
    # ITEM
    "item_total_views", "item_total_carts", "item_total_purchases",
    "item_cart_rate", "item_purchase_rate", "item_category",
    "item_days_since_first_seen", "item_popularity",
    # USER-ITEM
    "user_item_views", "user_item_carts", "user_item_purchases",
    "user_item_days_since_last_interaction", "category_affinity",
    # RANKING-ONLY
    "als_score", "itemcf_score", "candidate_source_code",
]

X = training_set[FEATURE_COLS]
y = training_set["label"]

xgb_params = dict(objective="binary:logistic", max_depth=5, n_estimators=200, learning_rate=0.1, random_state=42)
print("xgb_params:", xgb_params)

xgb_model = xgb.XGBClassifier(**xgb_params)
t0 = time.time()
xgb_model.fit(X, y)
xgb_train_time = time.time() - t0
print(f"\ntraining time: {xgb_train_time:.1f}s")

importances = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print("\nfeature importances:\n", importances)

xgb_params: {'objective': 'binary:logistic', 'max_depth': 5, 'n_estimators': 200, 'learning_rate': 0.1, 'random_state': 42}



training time: 2.9s

feature importances:
 item_total_carts                         0.202988
user_item_days_since_last_interaction    0.121111
itemcf_score                             0.110086
user_total_carts                         0.053695
als_score                                0.042673
user_interaction_span_days               0.038000
item_total_purchases                     0.034763
item_total_views                         0.033987
category_affinity                        0.032247
user_item_carts                          0.031795
user_item_views                          0.031404
user_top_category                        0.031177
user_days_since_last_event               0.031004
item_category                            0.030711
user_total_views                         0.027611
item_cart_rate                           0.024453
user_total_purchases                     0.023997
item_popularity                          0.023381
user_unique_items_viewed                 0.022136
item_d

## Cell 4: Rank candidates, produce Top-K
Score each user's ~50 candidates with the trained model, sort by predicted probability, keep the top 10. Save `final_recs_val.parquet` — the Phase 3 deliverable that Phase 4 evaluates against the popularity baseline.

In [4]:
import pickle

K = 10

training_set["xgb_score"] = xgb_model.predict_proba(X)[:, 1]

final_recs_val = (
    training_set.sort_values(["visitorid", "xgb_score"], ascending=[True, False])
    .groupby("visitorid").head(K)
    .copy()
)
final_recs_val["rank"] = final_recs_val.groupby("visitorid").cumcount() + 1
final_recs_val = final_recs_val[["visitorid", "itemid", "xgb_score", "rank"]]
final_recs_val.to_parquet(PROCESSED_DIR / "final_recs_val.parquet", index=False)

print("saved final_recs_val.parquet:", final_recs_val.shape)
print("users covered:", final_recs_val["visitorid"].nunique())
print("recs per user: mean={:.1f}, min={}, max={}".format(
    final_recs_val.groupby("visitorid").size().mean(),
    final_recs_val.groupby("visitorid").size().min(),
    final_recs_val.groupby("visitorid").size().max(),
))

print("\nsample final recommendations for 3 users:")
for uid in final_recs_val["visitorid"].unique()[:3]:
    sub = final_recs_val[final_recs_val["visitorid"] == uid]
    print(f"\n  user {uid}:")
    print(sub[["itemid", "xgb_score", "rank"]].to_string(index=False))

with open(PROCESSED_DIR / "xgboost_ranker.pkl", "wb") as f:
    pickle.dump(xgb_model, f)
print("\nsaved xgboost_ranker.pkl")

saved final_recs_val.parquet: (152400, 4)
users covered: 15240
recs per user: mean=10.0, min=10, max=10

sample final recommendations for 3 users:

  user 155:
 itemid  xgb_score  rank
 134620   0.001182     1
 123027   0.000978     2
 107229   0.000096     3
 304291   0.000077     4
 220749   0.000071     5
 442395   0.000055     6
 434684   0.000048     7
 191188   0.000042     8
  56209   0.000034     9
 216804   0.000030    10

  user 162:
 itemid  xgb_score  rank
 305656   0.001499     1
 390093   0.000543     2
 248862   0.000185     3
   1152   0.000159     4
 125906   0.000031     5
 368903   0.000022     6
  62206   0.000021     7
 262979   0.000020     8
 101561   0.000019     9
 112828   0.000018    10

  user 295:
 itemid  xgb_score  rank
  29196   0.000397     1
 312728   0.000145     2
 445817   0.000075     3
 122219   0.000056     4
 409804   0.000052     5
 381613   0.000024     6
 211796   0.000023     7
  46232   0.000021     8
 344723   0.000020     9
   9877   0.00